In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Reprezentarea calculatoarelor cuantice pentru transpiler


<details>
<summary><b>Versiunile pachetelor</b></summary>

Codul de pe această pagină a fost dezvoltat folosind cerințele de mai jos.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Pentru a converti un Circuit abstract într-un Circuit ISA care poate rula pe un QPU specific (unitate de procesare cuantică), transpiler-ul are nevoie de anumite informații despre QPU. Aceste informații se găsesc în două locuri: obiectul `BackendV2` (sau moștenitorul `BackendV1`) la care intenționezi să trimiți joburi, și atributul `Target` al backend-ului.

- [`Target`](../api/qiskit/qiskit.transpiler.Target) conține toate constrângerile relevante ale unui dispozitiv, cum ar fi Gate-urile de bază native, conectivitatea Qubit-urilor și informațiile de puls sau sincronizare.
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) deține un `Target` implicit, conține informații suplimentare -- cum ar fi [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), și oferă interfața pentru trimiterea joburilor cu Circuit-uri cuantice.

Poți, de asemenea, să furnizezi explicit informații pe care transpiler-ul să le folosească, de exemplu, dacă ai un caz de utilizare specific sau dacă crezi că aceste informații vor ajuta transpiler-ul să genereze un Circuit mai optimizat.

Precizia cu care transpiler-ul produce cel mai potrivit Circuit pentru hardware specific depinde de câte informații are `Target`-ul sau `Backend`-ul despre constrângerile sale.

> **Note:** Deoarece multe dintre algoritmii de transpilare subiacentă sunt stocastici, nu există nicio garanție că va fi găsit un Circuit mai bun.

Această pagină prezintă mai multe exemple de transmitere a informațiilor despre QPU către transpiler. Aceste exemple folosesc target-ul din backend-ul simulat [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Configurația implicită
Cea mai simplă utilizare a transpiler-ului este să furnizezi toate informațiile despre QPU prin furnizarea `Backend`-ului sau `Target`-ului. Pentru a înțelege mai bine cum funcționează transpiler-ul, construiește un Circuit și transpilează-l cu informații diferite, după cum urmează.

Importă bibliotecile necesare și instanțiază QPU-ul:
Pentru a converti un Circuit abstract într-un Circuit ISA care poate rula pe un procesor specific, transpiler-ul are nevoie de anumite informații despre procesor. De obicei, aceste informații sunt stocate în [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) sau [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) furnizat transpiler-ului și nu mai sunt necesare informații suplimentare. Cu toate acestea, poți furniza explicit informații pe care transpiler-ul să le folosească, de exemplu, dacă ai un caz de utilizare specific sau dacă crezi că aceste informații vor ajuta transpiler-ul să genereze un Circuit mai optimizat.

Acest subiect prezintă mai multe exemple de transmitere a informațiilor către transpiler. Aceste exemple folosesc target-ul din backend-ul simulat [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Circuit-ul exemplu folosește o instanță a [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) din biblioteca de Circuit-uri Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Acest exemplu folosește setările implicite pentru a transpila la `target`-ul `backend`-ului, care furnizează toate informațiile necesare pentru a converti Circuit-ul în unul care va rula pe backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Acest exemplu este folosit în secțiunile ulterioare ale acestui subiect pentru a ilustra că harta de cuplare și Gate-urile de bază sunt elementele esențiale de informații de transmis transpiler-ului pentru construcția optimă a Circuit-ului. QPU-ul poate de obicei să selecteze setări implicite pentru alte informații care nu sunt transmise, cum ar fi sincronizarea și planificarea.

## Harta de cuplare

Harta de cuplare este un graf care arată care Qubit-uri sunt conectate și, prin urmare, au Gate-uri cu două Qubit-uri între ele. Uneori acest graf este direcțional, ceea ce înseamnă că Gate-urile cu două Qubit-uri pot merge doar într-o singură direcție. Cu toate acestea, transpiler-ul poate întotdeauna să inverseze direcția unui Gate adăugând Gate-uri suplimentare cu un singur Qubit. Un Circuit cuantic abstract poate fi întotdeauna reprezentat pe acest graf, chiar dacă conectivitatea sa este limitată, prin introducerea de Gate-uri SWAP pentru a muta informațiile cuantice.

Qubit-urile din Circuit-urile noastre abstracte se numesc _Qubit-uri virtuale_, iar cele de pe harta de cuplare sunt _Qubit-uri fizice_. Transpiler-ul oferă o mapare între Qubit-urile virtuale și cele fizice. Unul dintre primii pași în transpilare, etapa _layout_, efectuează această mapare.

> **Note:** Deși etapa de rutare este împletită cu etapa _layout_ — care selectează Qubit-urile efective — în mod implicit, acest subiect le tratează ca etape separate pentru simplitate. Combinația de rutare și layout se numește _maparea Qubit-urilor_. Află mai multe despre aceste etape în subiectul [Etapele Transpiler-ului](transpiler-stages).

Transmite argumentul cheie `coupling_map` pentru a vedea efectul său asupra transpiler-ului:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Așa cum se arată mai sus, au fost inserate mai multe Gate-uri SWAP (fiecare constând din trei Gate-uri CX), ceea ce va cauza multe erori pe dispozitivele actuale. Pentru a vedea ce Qubit-uri sunt selectate pe topologia efectivă a Qubit-urilor, folosește `plot_circuit_layout` din Vizualizările Qiskit:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Aceasta arată că Qubit-urile noastre virtuale 0-11 au fost mapate trivial la linia de Qubit-uri fizice 0-11. Să revenim la setarea implicită (`optimization_level=1`), care folosește `VF2Layout` dacă este necesară rutare.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Acum nu mai sunt inserate Gate-uri SWAP, iar Qubit-urile fizice selectate sunt aceleași ca atunci când se folosește clasa `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Acum layout-ul este într-un inel. Deoarece acest layout respectă conectivitatea Circuit-ului, nu există Gate-uri SWAP, oferind un Circuit mult mai bun pentru execuție.

## Gate-uri de bază

Fiecare calculator cuantic acceptă un set limitat de instrucțiuni, numit _Gate-urile sale de bază_. Fiecare Gate din Circuit trebuie tradus în elementele acestui set. Acest set ar trebui să fie alcătuit din Gate-uri cu un singur Qubit și Gate-uri cu două Qubit-uri care furnizează un set universal de Gate-uri, ceea ce înseamnă că orice operație cuantică poate fi descompusă în acele Gate-uri. Aceasta este realizată de [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), iar Gate-urile de bază pot fi specificate ca argument cheie pentru transpiler pentru a furniza aceste informații.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Gate-urile implicite cu un singur Qubit pe _ibm_sherbrooke_ sunt `rz`, `x` și `sx`, iar Gate-ul implicit cu două Qubit-uri este `ecr` (rezonanță încrucișată ecouată). Gate-urile CX sunt construite din Gate-uri `ecr`, astfel că pe unele QPU-uri `ecr` este specificat ca Gate de bază cu două Qubit-uri, în timp ce pe altele `cx` este cel implicit. Gate-ul `ecr` este partea de _împletire_ a Gate-ului `cx`. Pe lângă Gate-urile de control, există și instrucțiuni de `delay` și `measurement`.

> **Note:** QPU-urile au Gate-uri de bază implicite, dar poți alege orice Gate-uri dorești, atât timp cât furnizezi instrucțiunea sau adaugi Gate-uri de puls (vezi [Crearea de pase pentru transpiler.](custom-transpiler-pass)) Gate-urile de bază implicite sunt cele pentru care au fost efectuate calibrări pe QPU, deci nu mai trebuie furnizate instrucțiuni/Gate-uri de puls suplimentare. De exemplu, pe unele QPU-uri `cx` este Gate-ul implicit cu două Qubit-uri, iar pe altele `ecr`. Vezi lista [gate-urilor și operațiunilor native posibile](/guides/qpu-information#native-gates) pentru mai multe detalii.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Rețineți că obiectele `CXGate` au fost descompuse în Gate-uri `ecr` și Gate-uri de bază cu un singur Qubit.
## Ratele de eroare ale dispozitivului
Clasa `Target` poate conține informații despre ratele de eroare pentru operațiunile de pe dispozitiv.
De exemplu, următorul cod preia proprietățile pentru Gate-ul de rezonanță încrucișată ecouată (ECR) dintre Qubit-ul 1 și 0 (rețineți că Gate-ul ECR este direcțional):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Rezultatul afișează durata Gate-ului (în secunde) și rata sa de eroare. Pentru a dezvălui informații despre erori transpiler-ului, construiește un model target cu `basis_gates` și `coupling_map` de mai sus și populează-l cu valori de eroare din backend-ul `FakeSherbrooke`.